# ✅ Generative adversarial networks (GAN)

In [1]:
# prerequisites
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bs = 128

# MNIST Dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=False)

# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=bs, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=bs, shuffle=False)

# Create output directory if it does not exist
os.makedirs("_output", exist_ok=True)

In [4]:
# ======================
# Models (Your Classes)
# ======================

class Generator(nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()
        self.fc1 = nn.Linear(g_input_dim, 256)
        self.fc2 = nn.Linear(256, 512)
        self.fc3 = nn.Linear(512, 1024)
        self.fc4 = nn.Linear(1024, g_output_dim)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.leaky_relu(self.fc3(x), 0.2)
        return torch.tanh(self.fc4(x))


class Discriminator(nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc3(x), 0.2)
        x = F.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))


# ======================
# Training Setup
# ======================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 100
image_dim = 28 * 28
batch_size = 128
epochs = 10
lr = 0.0002

# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # scale to [-1, 1]
])

dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize models
G = Generator(latent_dim, image_dim).to(device)
D = Discriminator(image_dim).to(device)

# Optimizers
optimizer_G = optim.Adam(G.parameters(), lr=lr)
optimizer_D = optim.Adam(D.parameters(), lr=lr)

criterion = nn.BCELoss()

# ======================
# Training Loop
# ======================

for epoch in range(epochs):
    for batch_idx, (real, _) in enumerate(dataloader):

        real = real.view(-1, image_dim).to(device)
        batch_size_curr = real.size(0)

        # Labels
        real_labels = torch.ones(batch_size_curr, 1).to(device)
        fake_labels = torch.zeros(batch_size_curr, 1).to(device)

        # ---------------------
        # Train Discriminator
        # ---------------------
        z = torch.randn(batch_size_curr, latent_dim).to(device)
        fake = G(z)

        D_real = D(real)
        D_fake = D(fake.detach())

        loss_D_real = criterion(D_real, real_labels)
        loss_D_fake = criterion(D_fake, fake_labels)

        loss_D = loss_D_real + loss_D_fake

        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()

        # ---------------------
        # Train Generator
        # ---------------------
        z = torch.randn(batch_size_curr, latent_dim).to(device)
        fake = G(z)

        D_fake = D(fake)
        loss_G = criterion(D_fake, real_labels)  # try to fool discriminator

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        if batch_idx == 0:
            print(f"Epoch [{epoch+1}/{epochs}] "
                  f"Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}")

    # ---------------------
    # Save sample images
    # ---------------------
    with torch.no_grad():
        z = torch.randn(64, latent_dim).to(device)
        generated = G(z).view(-1, 1, 28, 28)
        grid = utils.make_grid(generated, nrow=8, normalize=True)
        utils.save_image(grid, os.path.join("_output", f"generated_epoch_{epoch+1}.png"))

print("Training complete.")

Epoch [1/10] Loss D: 1.3860, Loss G: 0.6742
Epoch [2/10] Loss D: 0.1794, Loss G: 4.9853
Epoch [3/10] Loss D: 0.4962, Loss G: 3.5905
Epoch [4/10] Loss D: 0.7466, Loss G: 2.0494
Epoch [5/10] Loss D: 1.4761, Loss G: 1.0629
Epoch [6/10] Loss D: 0.6476, Loss G: 2.4691
Epoch [7/10] Loss D: 0.2960, Loss G: 3.1966
Epoch [8/10] Loss D: 0.4538, Loss G: 3.4870
Epoch [9/10] Loss D: 0.9197, Loss G: 2.7973
Epoch [10/10] Loss D: 0.5151, Loss G: 2.1617
Training complete.


# ✅ Deep convolutional generative adversarial networks (DCGAN)

## ↘️ TODO...

Write (from scratch) the two networks of a **DCGAN-style GAN** for **MNIST** images ($1\times 28 \times 28$), following the provided schema:

- **Generator**: $z \in \mathbb{R}^{n_z} \mapsto \hat{x} \in [-1,1]^{1\times 28 \times 28}$
- **Discriminator**: $x \mapsto \text{logit}$ (real vs fake)
- Training will use **`BCEWithLogitsLoss`**, so the discriminator must output **raw logits** (no sigmoid).

<br>🔹 **Generator $G$**

- A convolutional generator that upsamples:

\begin{align}
&(N, n_z, 1, 1)\\
\rightarrow &(N, 4n_{gf}, 7, 7)\\
\rightarrow &(N, 2n_{gf}, 14, 14)\\
\rightarrow &(N, n_{gf}, 28, 28)\\
\rightarrow &(N, 1, 28, 28)
\end{align}

- Constraints:

  - Use **ConvTranspose2d** for upsampling
  - Use **BatchNorm2d** + **ReLU**
  - Final activation must be **Tanh** so output is in $[-1,1]$

<br>🔹 **Discriminator/Critic $D$**

- A convolutional discriminator that downsamples:

\begin{align}
&(N, 1, 28, 28)\\
\rightarrow &(N, n_{df}, 14, 14)\\
\rightarrow &(N, 2n_{df}, 7, 7)\\
\rightarrow &(N, 4n_{df}, 7, 7)\\
\rightarrow &(N, 1)
\end{align}

- Constraints:

  - Use **Conv2d** for downsampling
  - Use **LeakyReLU(0.2)**
  - Use **BatchNorm2d** (except possibly on the first block)
  - The output must be a **logit** (no sigmoid)

<br>🔹 **Provided Skeleton**

- `Config` dataclass with `nz`, `ngf`, `ndf`
- `weights_init` function
- Training loop (optional, as “given code”)
- `Generator.__init__`, `Generator.forward`
- `Discriminator.__init__`, `Discriminator.forward`

<br>🔹 **Implement the Generator**

1. Create a `nn.Sequential` called `self.net`
2. Use the required blocks:
   - `ConvTranspose2d`
   - `BatchNorm2d`
   - `ReLU`
3. Ensure the tensor shapes match the plan:
   - from `1x1` to `7x7`, then `14x14`, then `28x28`
4. End with:
   - `nn.Conv2d(..., out_ch=1, ...)`
   - `nn.Tanh()`

- **Checkpoint:** `G(z).shape == (N, 1, 28, 28)`

<br>🔹 **Implement the Discriminator**

1. Create a `nn.Sequential` called `self.net`
2. Use the required blocks:
   - `Conv2d` with stride 2 for downsampling
   - `LeakyReLU(0.2)`
   - `BatchNorm2d` in deeper layers
3. End with a final conv producing `1x1` spatial output
4. Return a 1D tensor of logits:

$$
(N, 1, 1, 1) \to (N,)
$$

- **Checkpoint:** `D(x).shape == (N,)` and values are unconstrained logits.

<br>🔹 **Verify Compatibility with the Loss**

- Because we use:

```python
criterion = nn.BCEWithLogitsLoss()
```

- Real targets: tensor of ones
-	Fake targets: tensor of zeros
-	Discriminator output must be logits, not probabilities


In [5]:
## Overview

# This is a minimal, working **DCGAN-style GAN** in PyTorch for **MNIST** (images $1\times 28 \times 28$).

# - **Generator**: $z \in \mathbb{R}^{n_z} \mapsto \hat{x} \in [-1,1]^{1\times 28 \times 28}$
# - **Discriminator**: $x \mapsto \text{logit}$ (real vs fake)
# - Uses the standard **BCEWithLogitsLoss** (numerically stable).


import os
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils


# ----------------------------
# Config
# ----------------------------
@dataclass
class Config:
    data_root: str = "./data"
    out_dir: str = "./_output"
    batch_size: int = 128
    num_workers: int = 2
    nz: int = 100          # latent dim
    ngf: int = 64          # generator base channels
    ndf: int = 64          # discriminator base channels
    lr: float = 2e-4
    beta1: float = 0.5
    epochs: int = 10
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    sample_every: int = 500  # steps

cfg = Config()


# ----------------------------
# Initialization (DCGAN heuristic)
# ----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


In [6]:
# ----------------------------
# Models (DCGAN-ish for 28x28)
# ----------------------------
class Generator(nn.Module):
    """
    DCGAN-style generator:
    z -> (ngf*4)x7x7 -> upsample to 14x14 -> 28x28 -> 1 channel
    output in [-1, 1] via Tanh
    """
    def __init__(self, nz=100, ngf=64, out_ch=1):
        super().__init__()
        self.net = nn.Sequential(
            # z: (N, nz, 1, 1)
            # TODO: Initialize the first layer to get (N, ngf*4, 7, 7)
            torch.nn.ConvTranspose2d(nz, ngf * 4, kernel_size=7, stride=1, padding=0, bias=False),
            torch.nn.BatchNorm2d(ngf * 4),
            torch.nn.ReLU(inplace=True),
            # TODO: Initialize the second layer to get (N, ngf*2, 14, 14)
            torch.nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(ngf * 2),
            torch.nn.ReLU(inplace=True),
            # TODO: Initialize the third layer to get (N, ngf, 28, 28)
            torch.nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(ngf),
            torch.nn.ReLU(inplace=True),
            # TODO: Initialize the fourth layer to get (N, 1, 28, 28) in [-1, 1]
            torch.nn.ConvTranspose2d(ngf, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """
    DCGAN-style discriminator:
    1x28x28 -> logits (real/fake)
    No sigmoid here (use BCEWithLogitsLoss).
    """
    def __init__(self, ndf=64, in_ch=1):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: Initialize the first layer to get (N, ndf, 14, 14)
            torch.nn.Conv2d(in_ch, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.LeakyReLU(0.2, inplace=True),
            # TODO: Initialize the second layer to get (N, 2ndf, 7, 7)
            torch.nn.Conv2d(ndf, 2 * ndf, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(2 * ndf),
            torch.nn.LeakyReLU(0.2, inplace=True),
            # TODO: Initialize the third layer to get (N, 4ndf, 7, 7)
            torch.nn.Conv2d(2 * ndf, 4 * ndf, kernel_size=3, stride=1, padding=1, bias=False),
            torch.nn.BatchNorm2d(4 * ndf),
            torch.nn.LeakyReLU(0.2, inplace=True),
            # TODO: Initialize the fourth layer to get (N, 1, 1, 1)
            torch.nn.Conv2d(4 * ndf, 1, kernel_size=7, stride=1, padding=0, bias=False),
        )

    def forward(self, x):
        logits = self.net(x)
        return logits.view(x.size(0))  # (N,)

In [7]:
# ----------------------------
# Train
# ----------------------------

# MNIST in [-1, 1]
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # map to ~[-1, 1]
])

# dataset + dataloader
ds = datasets.MNIST(root=cfg.data_root, train=True, download=True, transform=tfm)
dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)

# models + optimizers
G = Generator(nz=cfg.nz, ngf=cfg.ngf).to(cfg.device)
D = Discriminator(ndf=cfg.ndf).to(cfg.device)
G.apply(weights_init)
D.apply(weights_init)

# loss 
criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, 0.999))
optD = optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, 0.999))

# fixed noise for sampling
fixed_z = torch.randn(64, cfg.nz, 1, 1, device=cfg.device)

# Create output directory if it does not exist
os.makedirs(cfg.out_dir, exist_ok=True)

# Training loop
step = 0
for epoch in range(cfg.epochs):
    for real, _ in dl:
        real = real.to(cfg.device, non_blocking=True)
        bsz = real.size(0)

        # --------------------
        # 1) Update Discriminator: maximize log D(x) + log(1 - D(G(z)))
        # --------------------
        z = torch.randn(bsz, cfg.nz, 1, 1, device=cfg.device)
        fake = G(z).detach()

        real_targets = torch.ones(bsz, device=cfg.device)
        fake_targets = torch.zeros(bsz, device=cfg.device)

        D_real = D(real)
        D_fake = D(fake)

        lossD_real = criterion(D_real, real_targets)
        lossD_fake = criterion(D_fake, fake_targets)
        lossD = lossD_real + lossD_fake

        optD.zero_grad(set_to_none=True)
        lossD.backward()
        optD.step()

        # --------------------
        # 2) Update Generator: maximize log D(G(z))  (non-saturating)
        # --------------------
        z = torch.randn(bsz, cfg.nz, 1, 1, device=cfg.device)
        fake = G(z)
        D_fake_for_G = D(fake)

        lossG = criterion(D_fake_for_G, real_targets)

        optG.zero_grad(set_to_none=True)
        lossG.backward()
        optG.step()

        # --------------------
        # Logging / samples
        # --------------------
        if step % 50 == 0:
            with torch.no_grad():
                d_real_prob = torch.sigmoid(D_real).mean().item()
                d_fake_prob = torch.sigmoid(D_fake).mean().item()
            print(
                f"epoch {epoch+1}/{cfg.epochs} step {step:06d} | "
                f"lossD {lossD.item():.4f} lossG {lossG.item():.4f} | "
                f"D(x) {d_real_prob:.3f} D(G(z)) {d_fake_prob:.3f}"
            )

        if step % cfg.sample_every == 0:
            with torch.no_grad():
                samples = G(fixed_z).cpu()
            grid = utils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
            utils.save_image(grid, os.path.join(cfg.out_dir, f"generated_DC_{step:06d}.png"))

        step += 1

print("Done.")


/home/filippo/code/gpu/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


epoch 1/10 step 000000 | lossD 1.9420 lossG 3.6819 | D(x) 0.831 D(G(z)) 0.768
epoch 1/10 step 000050 | lossD 0.1155 lossG 3.7950 | D(x) 0.963 D(G(z)) 0.056
epoch 1/10 step 000100 | lossD 0.4846 lossG 2.5599 | D(x) 0.821 D(G(z)) 0.206
epoch 1/10 step 000150 | lossD 1.5530 lossG 3.3255 | D(x) 0.974 D(G(z)) 0.716
epoch 1/10 step 000200 | lossD 0.4631 lossG 2.0869 | D(x) 0.836 D(G(z)) 0.205
epoch 1/10 step 000250 | lossD 1.7144 lossG 0.4827 | D(x) 0.248 D(G(z)) 0.006
epoch 1/10 step 000300 | lossD 0.3180 lossG 1.8516 | D(x) 0.876 D(G(z)) 0.137
epoch 1/10 step 000350 | lossD 0.2421 lossG 1.9802 | D(x) 0.860 D(G(z)) 0.073
epoch 1/10 step 000400 | lossD 0.2382 lossG 2.6220 | D(x) 0.857 D(G(z)) 0.063
epoch 1/10 step 000450 | lossD 0.3393 lossG 1.5071 | D(x) 0.775 D(G(z)) 0.040
epoch 2/10 step 000500 | lossD 0.3017 lossG 1.5062 | D(x) 0.796 D(G(z)) 0.037
epoch 2/10 step 000550 | lossD 0.3599 lossG 1.8603 | D(x) 0.787 D(G(z)) 0.059
epoch 2/10 step 000600 | lossD 0.9747 lossG 0.4525 | D(x) 0.599 

KeyboardInterrupt: 

# ✅ CycleGAN

## Minimal CycleGAN Example (PyTorch)

This is a **teaching-friendly** CycleGAN skeleton for **unpaired image-to-image translation**:

- Two domains: $A$ and $B$ (e.g., horses ↔ zebras)
- Two generators: $G_{A\to B}$ and $G_{B\to A}$
- Two discriminators: $D_A$ and $D_B$ (PatchGAN)
- Losses:
  - Adversarial (LSGAN): makes outputs look real in the target domain
  - Cycle-consistency: $A \to B \to A$ reconstructs the original
  - Identity (optional): preserves color/structure when input already in target domain

> NOTE: This is not a full production CycleGAN (no buffer/replay, no schedulers, no mixed precision). It’s meant to be **simple and readable**.

In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from torchvision.datasets import CIFAR10
import random

@dataclass
class CFG:
    # If using CIFAR classes rather than folders, provide dataset root
    use_cifar: bool = True
    cifar_root: str = "./data"
    classA: int = 0    # airplane
    classB: int = 1    # automobile

    out_dir: str = "./_cyclegan_out_cifar"
    image_size: int = 32
    batch_size: int = 16           # can increase if GPU memory allows
    num_workers: int = 4

    lr: float = 2e-4
    beta1: float = 0.5
    epochs: int = 200

    lambda_cyc: float = 10.0
    lambda_id: float = 5.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    sample_every: int = 500


class CIFARUnpairedDataset(torch.utils.data.Dataset):
    """
    Build two pools from CIFAR-10 classes classA and classB.
    Returns (imgA, imgB) as tensors transformed to [-1,1].
    """
    def __init__(self, root, classA, classB, transform, train=True, download=True):
        self.transform = transform
        ds = CIFAR10(root=root, train=train, download=download)
        self.imagesA = [ds.data[i] for i, lbl in enumerate(ds.targets) if lbl == classA]
        self.imagesB = [ds.data[i] for i, lbl in enumerate(ds.targets) if lbl == classB]
        if len(self.imagesA) == 0 or len(self.imagesB) == 0:
            raise ValueError("No images for chosen classes")
    def __len__(self):
        return max(len(self.imagesA), len(self.imagesB))
    def __getitem__(self, idx):
        a = Image.fromarray(self.imagesA[idx % len(self.imagesA)])
        b = Image.fromarray(random.choice(self.imagesB))
        return self.transform(a), self.transform(b)

# ----------------------------
# Data (unpaired)
# ----------------------------
class UnpairedImageDataset(Dataset):
    """
    Returns an unpaired sample (A_i, B_j).
    Domain folders should contain images.
    """
    def __init__(self, rootA, rootB, transform):
        self.pathsA = sorted([str(p) for p in Path(rootA).glob("*") if p.is_file()])
        self.pathsB = sorted([str(p) for p in Path(rootB).glob("*") if p.is_file()])
        if len(self.pathsA) == 0 or len(self.pathsB) == 0:
            raise ValueError("Domain folders must contain images.")
        self.t = transform

    def __len__(self):
        return max(len(self.pathsA), len(self.pathsB))

    def __getitem__(self, idx):
        pathA = self.pathsA[idx % len(self.pathsA)]
        pathB = self.pathsB[torch.randint(0, len(self.pathsB), (1,)).item()]  # random B

        imgA = Image.open(pathA).convert("RGB")
        imgB = Image.open(pathB).convert("RGB")

        return self.t(imgA), self.t(imgB)


# ----------------------------
# Model blocks
# ----------------------------
class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),

            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class ResnetGenerator(nn.Module):
    """
    ResNet-based generator used in CycleGAN.
    Input/Output: (N, 3, H, W), output in [-1, 1] via Tanh.
    """
    def __init__(self, in_ch=3, out_ch=3, n_filters=64, n_blocks=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, n_filters, 7),
            nn.InstanceNorm2d(n_filters),
            nn.ReLU(True),
        ]

        # Downsample
        c = n_filters
        for _ in range(2):
            layers += [
                nn.Conv2d(c, c * 2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(c * 2),
                nn.ReLU(True),
            ]
            c *= 2

        # ResNet blocks
        for _ in range(n_blocks):
            layers += [ResnetBlock(c)]

        # Upsample
        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(c, c // 2, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(c // 2),
                nn.ReLU(True),
            ]
            c //= 2

        # Output
        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(c, out_ch, 7),
            nn.Tanh(),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class PatchDiscriminator(nn.Module):
    """
    PatchGAN discriminator.
    Outputs a map of logits (N, 1, H', W') — we use LSGAN (MSE) loss.
    """
    def __init__(self, in_ch=3, n_filters=64):
        super().__init__()
        def block(in_c, out_c, stride):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1),
                nn.InstanceNorm2d(out_c),
                nn.LeakyReLU(0.2, inplace=True),
            )

        layers = [
            nn.Conv2d(in_ch, n_filters, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            block(n_filters, n_filters * 2, 2),
            block(n_filters * 2, n_filters * 4, 2),
            block(n_filters * 4, n_filters * 8, 1),

            nn.Conv2d(n_filters * 8, 1, 4, stride=1, padding=1),  # logits map
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ----------------------------
# Loss helpers
# ----------------------------
def lsgan_loss(pred, target_is_real: bool):
    """
    LSGAN: MSE between discriminator logits-map and target labels (1 or 0).
    """
    target_val = 1.0 if target_is_real else 0.0
    target = torch.full_like(pred, fill_value=target_val)
    return torch.mean((pred - target) ** 2)


# ----------------------------
# Train
# ----------------------------
def main(cfg: CFG):
    torch.manual_seed(cfg.seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    # Transform: resize/crop + normalize to [-1,1]
    t = transforms.Compose([
        transforms.Resize(cfg.image_size),
        transforms.CenterCrop(cfg.image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),
                             (0.5, 0.5, 0.5)),
    ])

    ds = UnpairedImageDataset(cfg.rootA, cfg.rootB, t)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, pin_memory=True)

    # Models
    G_AB = ResnetGenerator(n_blocks=9).to(cfg.device)
    G_BA = ResnetGenerator(n_blocks=9).to(cfg.device)
    D_A = PatchDiscriminator().to(cfg.device)
    D_B = PatchDiscriminator().to(cfg.device)

    # Optims
    opt_G = optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()),
                       lr=cfg.lr, betas=(cfg.beta1, 0.999))
    opt_D = optim.Adam(list(D_A.parameters()) + list(D_B.parameters()),
                       lr=cfg.lr, betas=(cfg.beta1, 0.999))

    l1 = nn.L1Loss()

    step = 0
    for epoch in range(cfg.epochs):
        for realA, realB in dl:
            realA = realA.to(cfg.device, non_blocking=True)
            realB = realB.to(cfg.device, non_blocking=True)

            # -----------------------------------------
            # (1) Train Generators: G_AB and G_BA
            # -----------------------------------------
            fakeB = G_AB(realA)
            fakeA = G_BA(realB)

            recA = G_BA(fakeB)  # A -> B -> A
            recB = G_AB(fakeA)  # B -> A -> B

            # GAN losses (LSGAN)
            loss_GAN_AB = lsgan_loss(D_B(fakeB), True)
            loss_GAN_BA = lsgan_loss(D_A(fakeA), True)

            # Cycle-consistency
            loss_cyc = l1(recA, realA) + l1(recB, realB)

            # Identity loss (optional)
            if cfg.lambda_id > 0:
                idA = G_BA(realA)  # should be ~realA
                idB = G_AB(realB)  # should be ~realB
                loss_id = l1(idA, realA) + l1(idB, realB)
            else:
                loss_id = torch.tensor(0.0, device=cfg.device)

            loss_G = (loss_GAN_AB + loss_GAN_BA) \
                     + cfg.lambda_cyc * loss_cyc \
                     + cfg.lambda_id * loss_id

            opt_G.zero_grad(set_to_none=True)
            loss_G.backward()
            opt_G.step()

            # -----------------------------------------
            # (2) Train Discriminators: D_A and D_B
            # -----------------------------------------
            with torch.no_grad():
                fakeB_det = fakeB.detach()
                fakeA_det = fakeA.detach()

            # D_A: realA vs fakeA
            loss_DA = 0.5 * (lsgan_loss(D_A(realA), True) + lsgan_loss(D_A(fakeA_det), False))
            # D_B: realB vs fakeB
            loss_DB = 0.5 * (lsgan_loss(D_B(realB), True) + lsgan_loss(D_B(fakeB_det), False))

            loss_D = loss_DA + loss_DB

            opt_D.zero_grad(set_to_none=True)
            loss_D.backward()
            opt_D.step()

            if step % 50 == 0:
                print(
                    f"epoch {epoch+1}/{cfg.epochs} step {step:06d} | "
                    f"lossG {loss_G.item():.3f} (gan {loss_GAN_AB.item()+loss_GAN_BA.item():.3f}, "
                    f"cyc {loss_cyc.item():.3f}, id {loss_id.item():.3f}) | "
                    f"lossD {loss_D.item():.3f}"
                )

            if step % cfg.sample_every == 0:
                # Save a small visual: realA, fakeB, recA and realB, fakeA, recB
                with torch.no_grad():
                    grid = torch.cat([realA[:1], fakeB[:1], recA[:1],
                                      realB[:1], fakeA[:1], recB[:1]], dim=0)
                    # unnormalize for saving: torchvision save_image handles normalize=True nicely
                    save_image(grid, os.path.join(cfg.out_dir, f"sample_{step:06d}.png"),
                               nrow=3, normalize=True, value_range=(-1, 1))

            step += 1

    print("Training complete.")

In [11]:
cfg = CFG()
# in main() replace transform with:
t = transforms.Compose([
    transforms.Resize(cfg.image_size),
    transforms.CenterCrop(cfg.image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),  # CIFAR-10 mean
                         (0.2470, 0.2435, 0.2616))  # CIFAR-10 std
])


ds = CIFARUnpairedDataset(cfg.cifar_root, cfg.classA, cfg.classB, t, train=True)
dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)

In [12]:
main(cfg)